In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets
from torchvision import transforms

D:\ANACONDA\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
#PREPROCESSING
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.1307,),
        (0.3081,)
    )
])

In [5]:
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)


Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [02:19<00:00, 70.9kB/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 150kB/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:09<00:00, 182kB/s]


Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 657kB/s]

Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw



In [9]:
train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader=DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [40]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1=nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.conv2=nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.pool=nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
        
        self.fc1 = nn.Linear(
            32*7*7,
            128
        )

        self.fc2 = nn.Linear(
            128,
            10
        )

    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        x=torch.flatten(x,start_dim=1)
        x=F.relu(self.fc1(x))
        x=self.fc2(x)
        
        return x

In [43]:
model=CNN()

In [45]:
criterion=nn.CrossEntropyLoss()

In [47]:
optimiser=optim.Adam(model.parameters(),lr=0.01)

In [49]:
epochs=5
for epoch in range(epochs):
    epoch_loss=0
    model.train()
    for x_train,y_train in train_loader:
        optimiser.zero_grad()
        output=model(x_train)
        loss=criterion(output,y_train)
        loss.backward()
        optimiser.step()
        epoch_loss+=loss.item()

    print(
        f"Epoch {epoch+1}: {epoch_loss:.4f}"
    )

Epoch 1: 168.1051
Epoch 2: 84.5027
Epoch 3: 77.6531
Epoch 4: 75.8624
Epoch 5: 68.6527


In [51]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images,labels in test_loader:

        outputs = model(images)

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

In [53]:
accuracy = 100 * correct / total

print(
    f"Accuracy: {accuracy:.2f}%"
)

Accuracy: 97.70%
